In [1]:
import math
import numpy as np
import pandas as pd
import copy
import re
import ast
from collections import Counter, defaultdict
from itertools import combinations
from numpy.linalg import det
from tqdm import tqdm
from scipy.stats import chi2
from statsmodels.stats.contingency_tables import mcnemar

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## Load Initial Policy CSVs and Perform Data Processing

In [2]:
# load all CSVs
temp_df_qwen = pd.read_csv('InitialPolicyData/botdetect_policy_df_Qwen25_7B.csv')
temp_df_mistral = pd.read_csv('InitialPolicyData/botdetect_policy_df_Mistral_7B.csv')
temp_df_gemma = pd.read_csv('InitialPolicyData/botdetect_policy_df_gemma_7b.csv')
temp_df_yi = pd.read_csv('InitialPolicyData/botdetect_policy_df_Yi_9B.csv')
temp_df_granite = pd.read_csv('InitialPolicyData/botdetect_policy_df_granite_8b.csv')
temp_df_zephyr = pd.read_csv('InitialPolicyData/botdetect_policy_df_zephyr_7B.csv')
temp_df_deepseek_llm = pd.read_csv('InitialPolicyData/botdetect_policy_df_deepseek_llm_7b.csv')
temp_df_deepseek_qwen = pd.read_csv('InitialPolicyData/botdetect_policy_df_deepseek_r1_qwen_7b.csv')

In [3]:
# define mapping so we can rename columns 
COMMON_COLUMNS = ["disc_init_policy", "gen_init_policy", "answerKey", "disc_answer", "gen_answer"]
RENAME_MAPPING = {"disc_answer": "ED_consensus", "answerKey":"answer_letter", "gen_answer": "EG_consensus"}

In [4]:
# rename columns same way original did
def process_dataframe(temp_df, COMMON_COLUMNS = COMMON_COLUMNS, RENAME_MAPPING = RENAME_MAPPING):
    df = temp_df[COMMON_COLUMNS].copy()
    df.rename(columns=RENAME_MAPPING, inplace=True)
    return df

In [5]:
# process dataframes
df_qwen = process_dataframe(temp_df_qwen)
df_mistral = process_dataframe(temp_df_mistral)
df_gemma = process_dataframe(temp_df_gemma)
df_yi = process_dataframe(temp_df_yi)
df_granite = process_dataframe(temp_df_granite)
df_zephyr = process_dataframe(temp_df_zephyr)
df_deepseek_llm = process_dataframe(temp_df_deepseek_llm)
df_deepseek_qwen = process_dataframe(temp_df_deepseek_qwen)

In [6]:
# clean strings
def parse_np_float_string(s):
    if isinstance(s, dict):
        return s
    if not isinstance(s, str):
        return {}

    try:
        s_clean = re.sub(r'np\.float32\((.*?)\)', r'\1', s)
        return ast.literal_eval(s_clean)
    except Exception as e:
        print("Still failed to parse:", s)
        return {}

In [7]:
# define all dfs for processing
all_dfs = [df_qwen,
           df_mistral,
           df_gemma,
           df_yi,
           df_granite,
           df_zephyr,
           df_deepseek_llm,
           df_deepseek_qwen]

In [8]:
# strip np.float32 wrappers
for df in all_dfs:
    df["disc_init_policy"] = df["disc_init_policy"].apply(parse_np_float_string)
    df["gen_init_policy"]  = df["gen_init_policy"].apply(parse_np_float_string)

In [9]:
# use fallback if literal_eval are still strings
for df in all_dfs:
    for col in ["disc_init_policy", "gen_init_policy"]:
        if df[col].dtype == object and isinstance(df[col].iloc[0], str):
            df[col] = df[col].apply(ast.literal_eval)

## Normalize Discriminator Policies (Softmax Refinement)

In [10]:
# define softmax function as helper for apply_softmax
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

In [11]:
# applies softmax across all choices for correct and incorrect separately
# output: A: {correct: p1, incorrect: p2}, B:{correct: p3, incorrect: p4} (same as input)
def apply_softmax(choice_dict):

    correct_values = np.array([v['correct']   for v in choice_dict.values()])
    incorrect_values = np.array([v['incorrect'] for v in choice_dict.values()])

    softmax_correct = softmax(correct_values)
    softmax_incorrect = softmax(incorrect_values)

    result = {}
    for i, choice in enumerate(choice_dict.keys()):
        result[choice] = {
            'correct': float(softmax_correct[i]),
            'incorrect': float(softmax_incorrect[i])
        }
    return result

In [12]:
for df in all_dfs:
    df["disc_init_policy_refine"] = df["disc_init_policy"].apply(apply_softmax)

## Extract Initial Answers From Policies

In [13]:
# returns argmax choice by correct probability
def get_most_probable_letter(answer_dict):
    if answer_dict is None:
        return None
    return max(answer_dict.items(), key = lambda item: item[1]['correct'])[0]

In [14]:
for df in all_dfs:
    df["disc_init_answer"] = df["disc_init_policy"].apply(get_most_probable_letter)
    df["disc_init_refine_answer"] = df["disc_init_policy_refine"].apply(get_most_probable_letter)

In [15]:
print("Disc init answer distributions:")
for name, df in [("Qwen25", df_qwen),
                 ("Mistral", df_mistral),
                 ("Gemma", df_gemma),
                 ("Yi", df_yi),
                 ("Granite", df_granite),
                 ("Zephyr", df_zephyr),
                 ("DeepSeek-LLM", df_deepseek_llm),
                 ("DeepSeek-R1-Qwen", df_deepseek_qwen)]:
    counts = df['disc_init_answer'].value_counts()
    true_counts = df['answer_letter'].value_counts()
    total  = len(df)
    print(f"{name}: A = {counts.get('A', 0)} ({counts.get('A', 0) / total*100:.2f}%)\n"
          f"{name}: B = {counts.get('B', 0)} ({counts.get('B', 0) / total*100:.2f}%)\n"
          f"{name} Accuracy: {(df['disc_init_answer'] == df['answer_letter']).mean()}\n\n")

Disc init answer distributions:
Qwen25: A = 8992 (89.92%)
Qwen25: B = 1008 (10.08%)
Qwen25 Accuracy: 0.482


Mistral: A = 9112 (91.12%)
Mistral: B = 888 (8.88%)
Mistral Accuracy: 0.4772


Gemma: A = 9914 (99.14%)
Gemma: B = 86 (0.86%)
Gemma Accuracy: 0.502


Yi: A = 6924 (69.24%)
Yi: B = 3076 (30.76%)
Yi Accuracy: 0.5266


Granite: A = 2792 (27.92%)
Granite: B = 7208 (72.08%)
Granite Accuracy: 0.5524


Zephyr: A = 9995 (99.95%)
Zephyr: B = 5 (0.05%)
Zephyr Accuracy: 0.5003


DeepSeek-LLM: A = 8763 (87.63%)
DeepSeek-LLM: B = 1237 (12.37%)
DeepSeek-LLM Accuracy: 0.4639


DeepSeek-R1-Qwen: A = 9388 (93.88%)
DeepSeek-R1-Qwen: B = 612 (6.12%)
DeepSeek-R1-Qwen Accuracy: 0.483




## Mutual Information Reweighing

In [16]:
# reweighs joint probabilities
# outputs label: p_gen * p_disc for y in {A, B}, label in {correct, incorrect}
def reweight_joint(row):
    plm_y_given_xl = row['gen_init_policy']
    plm_l_given_xy = row['disc_init_policy']

    new_joint = {}
    for y, label_probs in plm_y_given_xl.items():
        new_joint[y] = {}
        for label, p1 in label_probs.items():
            p2 = plm_l_given_xy.get(label, {}).get(y, 0.0)
            new_joint[y][label] = p1 * p2
    return new_joint

In [17]:
# argmax over correct key of MI dict
def get_max_correct_label(mi_dict):
    return max(mi_dict['correct'], key = mi_dict['correct'].get)

In [18]:
for df in all_dfs:
    df["MI"] = df.apply(reweight_joint, axis=1)
    df["MI_answer"] = df["MI"].apply(get_max_correct_label)

## Majority Vote Evaluation

In [19]:
# computers per-model accuracy and majority vote acuracy
# majority threshold: floor(n_models / 2)
def evaluate_majority_vote(
    model_dfs: dict,
    answer_col: str = 'disc_init_refine_answer',
    label_col: str = 'answer_letter'
):
    model_names = list(model_dfs.keys())
    if not model_names:
        raise ValueError("model_dfs cannot be empty.")

    num_models = len(model_names)

    # combine predictions; uses first model as a base
    base_df = model_dfs[model_names[0]]

    combined_df = pd.DataFrame({
        "question_id": base_df.index,
        "correct_answer": base_df[label_col]
    })

    # add each model's prediction as a new column
    pred_cols = []
    for name, df in model_dfs.items():
        col = f"{name}_pred"
        combined_df[col] = df[answer_col].values
        pred_cols.append(col)
        
    # define majority vote function to calculate majority vote
    def get_majority_vote(row):
        
        # majority threshold is floor(n_models / 2)
        majority_threshold = math.floor(num_models / 2) + 1
        
        votes = [row[col] for col in pred_cols if pd.notna(row[col])]
        
        if not votes:
            return None
            
        vote_counts = Counter(votes)
        
        most_common_ans, highest_count = vote_counts.most_common(1)[0]
        
        # return answer only if it meets majority threshold
        return most_common_ans if highest_count >= majority_threshold else None

    # run function to get majority vote for each prompt
    combined_df["majority_vote"] = combined_df.apply(get_majority_vote, axis=1)

    # check correctness
    for name in model_names:
        combined_df[f"{name}_correct"] = combined_df[f"{name}_pred"] == combined_df["correct_answer"]
    combined_df["is_majority_correct"] = combined_df["majority_vote"] == combined_df["correct_answer"]

    # summarize results
    total_questions = len(combined_df)
    
    metrics    = ["Total Questions"]
    counts     = [total_questions]
    accuracies = ["100%"]

    # add results for each individual model
    for name in model_names:
        correct_count = combined_df[f"{name}_correct"].sum()
        metrics.append(f"{name} Accuracy")
        counts.append(correct_count)
        accuracies.append(f"{(correct_count / total_questions) * 100:.2f}%")

    # add final majority vote result
    majority_correct = combined_df["is_majority_correct"].sum()
    majority_threshold_display = math.floor(num_models / 2)
    metrics.append(f"Majority Vote (>{majority_threshold_display} models) Accuracy")
    counts.append(majority_correct)
    accuracies.append(f"{(majority_correct / total_questions) * 100:.2f}%")

    results_df = pd.DataFrame({"Metric": metrics, "Count": counts, "Accuracy": accuracies})
    return combined_df, results_df

## Answer Alignment Evaluation

In [20]:
# gets majority vote; majority threshold: floor(n_models / 2)
def evaluate_majority_vote(
    model_dfs: dict,
    answer_col: str = 'disc_init_refine_answer',
    label_col:  str = 'answer_letter'
):
    model_names = list(model_dfs.keys())
    if not model_names:
        raise ValueError("model_dfs cannot be empty.")

    num_models = len(model_names)
    base_df    = model_dfs[model_names[0]]

    combined_df = pd.DataFrame({
        "question_id": base_df.index,
        "correct_answer": base_df[label_col]
    })

    pred_cols = []
    for name, df in model_dfs.items():
        col = f"{name}_pred"
        combined_df[col] = df[answer_col].values
        pred_cols.append(col)

    def get_majority_vote(row):
        majority_threshold = math.floor(num_models / 2) + 1
        votes = [row[col] for col in pred_cols if pd.notna(row[col])]
        if not votes:
            return None
        vote_counts = Counter(votes)
        most_common_ans, highest_count = vote_counts.most_common(1)[0]
        return most_common_ans if highest_count >= majority_threshold else None

    combined_df["majority_vote"] = combined_df.apply(get_majority_vote, axis=1)

    for name in model_names:
        combined_df[f"{name}_correct"] = combined_df[f"{name}_pred"] == combined_df["correct_answer"]
    combined_df["is_majority_correct"] = combined_df["majority_vote"] == combined_df["correct_answer"]

    total_questions = len(combined_df)
    metrics = ["Total Questions"]
    counts = [total_questions]
    accuracies = ["100%"]

    for name in model_names:
        correct_count = combined_df[f"{name}_correct"].sum()
        metrics.append(f"{name} Accuracy")
        counts.append(correct_count)
        accuracies.append(f"{(correct_count / total_questions) * 100:.2f}%")

    majority_correct = combined_df["is_majority_correct"].sum()
    majority_threshold_display = math.floor(num_models / 2)
    metrics.append(f"Majority Vote (>{majority_threshold_display} models) Accuracy")
    counts.append(majority_correct)
    accuracies.append(f"{(majority_correct / total_questions) * 100:.2f}%")

    results_df = pd.DataFrame({"Metric": metrics, "Count": counts, "Accuracy": accuracies})
    return combined_df, results_df

In [21]:
# measures consensus; all models MUST agree (not just majority)
# if PEG works, unanimous agreement should increase post-update while accuracy given unanimous agreement stays high
def evaluate_answer_alignment(
    model_dfs: dict,
    answer_col: str = 'disc_init_refine_answer',
    label_col: str = 'answer_letter'
):
    model_names = list(model_dfs.keys())
    base_df = model_dfs[model_names[0]]
    total_n = len(base_df)

    preds = pd.DataFrame({
        name: model_dfs[name][answer_col].values
        for name in model_names
    })
    correct_answers = base_df[label_col].values

    # Per-model correctness
    per_model_correct = {
        name: (preds[name].values == correct_answers).sum()
        for name in model_names
    }

    # Unanimous agreement: all models predict the same label
    unanimous_mask  = preds.nunique(axis=1) == 1
    unanimous_count = unanimous_mask.sum()

    # Unanimous AND correct
    unanimous_answer        = preds[unanimous_mask].iloc[:, 0]
    unanimous_correct_count = (
        unanimous_answer.values == correct_answers[unanimous_mask]
    ).sum()

    metrics = ["Total Samples"]
    counts = [total_n]
    accuracies = ["100%"]

    for name in model_names:
        metrics.append(f"{name} Accuracy")
        counts.append(per_model_correct[name])
        accuracies.append(f"{per_model_correct[name]/total_n*100:.2f}%")

    metrics.append("Unanimous Agreement Rate")
    counts.append(unanimous_count)
    accuracies.append(f"{unanimous_count/total_n*100:.2f}%")

    metrics.append("Unanimous & Correct Rate")
    counts.append(unanimous_correct_count)
    accuracies.append(f"{unanimous_correct_count/total_n*100:.2f}%")

    if unanimous_count > 0:
        metrics.append("Accuracy Given Unanimous Agreement")
        counts.append(unanimous_correct_count)
        accuracies.append(f"{unanimous_correct_count/unanimous_count*100:.2f}%")

    return pd.DataFrame({"Metric": metrics, "Count": counts, "Accuracy": accuracies})

## Discriminator Class and DMI Reward Functions

In [22]:
np.random.seed(213)
label_to_index = {'correct': 0, 'incorrect': 1}

In [23]:
# initialize discriminator class; no changes made
class Discriminator:

    def __init__(self, init_policy):
        """Initialize with given probability dictionary."""
        self.policy = init_policy

    def respond(self, choice, max_choice):
        """Sample a response ('correct' or 'incorrect') for a given answer choice."""
        return "correct" if choice == max_choice else "incorrect"

    def log_gradient(self, choice, response):
        
        if response == "correct":
            grad_correct =  1 / self.policy[choice]['correct']
            grad_incorrect = -1 / self.policy[choice]['correct']
        else:
            grad_correct = -1 / self.policy[choice]['incorrect']
            grad_incorrect = 1 / self.policy[choice]['incorrect']
        return {'correct': grad_correct, 'incorrect': grad_incorrect}

    def get_max_correct_choice(self):
        """Return the choice letter with the highest 'correct' probability."""
        return max(self.policy.items(), key = lambda x: x[1]['correct'])[0]

    def update_policy(self, choice, response, reward, learning_rate=0.1):
        grads = self.log_gradient(choice, response)
        prob_correct = self.policy[choice]['correct']
        prob_incorrect = self.policy[choice]['incorrect']

        log_prob_correct = np.log(prob_correct) + learning_rate * reward * grads['correct']
        log_prob_incorrect = np.log(prob_incorrect) + learning_rate * reward * grads['incorrect']

        max_log = max(log_prob_correct, log_prob_incorrect)
        log_prob_correct -= max_log
        log_prob_incorrect -= max_log

        exp_correct = np.exp(log_prob_correct)
        exp_incorrect = np.exp(log_prob_incorrect)
        total = exp_correct + exp_incorrect

        self.policy[choice]['correct']   = np.clip(exp_correct / total, 1e-6, 1 - 1e-6)
        self.policy[choice]['incorrect'] = 1 - self.policy[choice]['correct']

In [24]:
# z-score to normalize DMI rewards
def normalize_rewards(dmi_dict):
    values = np.array(list(dmi_dict.values()))
    mean = np.mean(values)
    std = np.std(values) + 1e-8
    return {k: (v - mean) / std for k, v in dmi_dict.items()}

In [25]:
# builds 2x2 co-report matrix for discriminators i and j
def generate_matrix(part_data, i, j):
    matrix = np.zeros((2, 2), dtype=int)
    for row in part_data:
        row_i = label_to_index[row[i]]
        row_j = label_to_index[row[j]]
        matrix[row_i][row_j] += 1
    return matrix

In [26]:
# calculate DMI score = det(M1) * det(M2)
def calculate_dmi_score(part1_matrix, part2_matrix):
    return np.linalg.det(part1_matrix) * np.linalg.det(part2_matrix)

In [27]:
# splits back in half, builds co-report matrices for each pair of discriminators, and sums DMI scores per discriminator
def compute_dmi_payments(responses):
    """Compute DMI payments using determinants of agreement matrices."""
    n      = len(responses)
    split1 = responses[:(n + 1) // 2]
    split2 = responses[(n + 1) // 2:]

    n_d        = len(responses[0])
    d_dmi_dict = {i: 0 for i in range(n_d)}

    for i, j in combinations(range(n_d), 2):
        part1_matrix = generate_matrix(split1, i, j)
        part2_matrix = generate_matrix(split2, i, j)
        dmi_score    = calculate_dmi_score(part1_matrix, part2_matrix)
        d_dmi_dict[i] += dmi_score
        d_dmi_dict[j] += dmi_score

    return d_dmi_dict

## Batch Update Discriminators

In [28]:
# batch update discriminator policies for a variable number of discriminators
def batch_update_discriminators(
    discriminator_dfs: dict,
    choice_df,
    T_steps,
    batch_size = 8,
    learning_rate = 0.1
):
    if batch_size < 4:
        print("Warning: batch_size should be >= 4 for DMI calculation.")

    # extract model names and list of dataframes for indexed processing
    model_names = list(discriminator_dfs.keys())
    df_list = [df.copy() for df in discriminator_dfs.values()]
    
    num_discriminators = len(df_list)
    n_rows = df_list[0].shape[0]

    # initialize the column for updated policies in each dataframe
    for df in df_list:
        df["updated_disc_policy"] = None

    for i in range(0, n_rows, batch_size):
        current_batch_size = min(batch_size, n_rows - i)
        if current_batch_size < 4:
            continue

        # initialize discriminators from refined policies
        discriminators = [
            [
                Discriminator(copy.deepcopy(df.loc[i + j, 'disc_init_policy_refine']))
                for j in range(current_batch_size)
            ]
            for df in df_list
        ]

        # select choice for each task using choice_df's gen_init_policy
        choices = [
            max(
                choice_df.loc[i + j, "gen_init_policy"]["correct"],
                key = choice_df.loc[i + j, "gen_init_policy"]["correct"].get
            )
            for j in range(current_batch_size)
        ]

        for _ in range(T_steps):
            # collect responses from all discriminators for all tasks in the batch
            batch_responses = []
            for j in range(current_batch_size):
                task_responses = []
                for d in range(num_discriminators):
                    max_choice = discriminators[d][j].get_max_correct_choice()
                    response = discriminators[d][j].respond(choices[j], max_choice)
                    task_responses.append(response)
                batch_responses.append(task_responses)

            # compute and normalize DMI payments
            dmi_scores = compute_dmi_payments(batch_responses)
            dmi_scores = normalize_rewards(dmi_scores)

            # update each discriminator's policy
            for d in range(num_discriminators):
                for j in range(current_batch_size):
                    choice   = choices[j]
                    response = batch_responses[j][d]
                    reward   = dmi_scores.get(d, 0)
                    discriminators[d][j].update_policy(choice, response, reward, learning_rate)

        # save updated policies back to DataFrames
        for d in range(num_discriminators):
            for j in range(current_batch_size):
                df_list[d].at[i + j, "updated_disc_policy"] = discriminators[d][j].policy

    # extract final answer from updated policy
    for df in df_list:
        df['updated_answer_letter'] = df['updated_disc_policy'].apply(get_most_probable_letter)

    return {name: df for name, df in zip(model_names, df_list)}

## Baseline Evaluation

In [38]:
# 3-model configuration
discriminator_3dict = {
    "Qwen25_7B": df_qwen,
    "Mistral_7B": df_mistral,
    "Gemma_7B": df_gemma,
}

combined_3_init_dis, results_3_init_dis = evaluate_majority_vote(
    model_dfs = discriminator_3dict,
    answer_col = 'disc_init_answer',
    label_col = 'answer_letter'
)
print("3-Model: Initial Discriminator Policy (raw)")
display(results_3_init_dis)

combined_3_init_refine, results_3_init_refine = evaluate_majority_vote(
    model_dfs = discriminator_3dict,
    answer_col = 'disc_init_refine_answer',
    label_col = 'answer_letter'
)
print("\n3-Model: Initial Discriminator Policy (softmax-refined)")
display(results_3_init_refine)

combined_3_ED, results_3_ED = evaluate_majority_vote(
    model_dfs = discriminator_3dict,
    answer_col = 'ED_consensus',
    label_col = 'answer_letter'
)
print("\n3-Model: Consensus Game Policy (ED_consensus)")
display(results_3_ED)

print("\n3-Model: Alignment Analysis (Init Discriminator)")
display(evaluate_answer_alignment(
    discriminator_3dict,
    answer_col = 'disc_init_refine_answer',
    label_col = 'answer_letter'
))

3-Model: Initial Discriminator Policy (raw)


,Metric,Count,Accuracy
0,Total Questions,10000,100%
1,Qwen25_7B Accuracy,4820,48.20%
2,Mistral_7B Accuracy,4772,47.72%
3,Gemma_7B Accuracy,5020,50.20%
4,Majority Vote (>1 models) Accuracy,4923,49.23%



3-Model: Initial Discriminator Policy (softmax-refined)


,Metric,Count,Accuracy
0,Total Questions,10000,100%
1,Qwen25_7B Accuracy,4820,48.20%
2,Mistral_7B Accuracy,4772,47.72%
3,Gemma_7B Accuracy,5020,50.20%
4,Majority Vote (>1 models) Accuracy,4923,49.23%



3-Model: Consensus Game Policy (ED_consensus)


,Metric,Count,Accuracy
0,Total Questions,10000,100%
1,Qwen25_7B Accuracy,4820,48.20%
2,Mistral_7B Accuracy,4772,47.72%
3,Gemma_7B Accuracy,5020,50.20%
4,Majority Vote (>1 models) Accuracy,4923,49.23%



3-Model: Alignment Analysis (Init Discriminator)


,Metric,Count,Accuracy
0,Total Samples,10000,100%
1,Qwen25_7B Accuracy,4820,48.20%
2,Mistral_7B Accuracy,4772,47.72%
3,Gemma_7B Accuracy,5020,50.20%
4,Unanimous Agreement Rate,8299,82.99%
5,Unanimous & Correct Rate,3994,39.94%
6,Accuracy Given Unanimous Agreement,3994,48.13%


In [39]:
# 6-model configuration
discriminator_6dict = {
    "Qwen25_7B": df_qwen,
    "Mistral_7B": df_mistral,
    "Gemma_7B": df_gemma,
    "Yi_9B": df_yi,
    "Granite_8B": df_granite,
    "Zephyr_7B": df_zephyr,
}

# 7-model configuration
discriminator_7dict = {
    "Qwen25_7B": df_qwen,
    "Mistral_7B": df_mistral,
    "Gemma_7B": df_gemma,
    "Yi_9B": df_yi,
    "Granite_8B": df_granite,
    "Zephyr_7B": df_zephyr,
    "DeepSeek_LLM": df_deepseek_llm,
}

# 8-model configuration
discriminator_8dict = {
    "Qwen25_7B": df_qwen,
    "Mistral_7B": df_mistral,
    "Gemma_7B": df_gemma,
    "Yi_9B": df_yi,
    "Granite_8B": df_granite,
    "Zephyr_7B": df_zephyr,
    "DeepSeek_LLM": df_deepseek_llm,
    "DeepSeek_R1_Qwen": df_deepseek_qwen,
}

for label, d in [("6-Model", discriminator_6dict),
                 ("7-Model", discriminator_7dict),
                 ("8-Model", discriminator_8dict)]:
    _, res = evaluate_majority_vote(
        model_dfs=d, answer_col = 'disc_init_answer', label_col = 'answer_letter'
    )
    print(f"{label}: Initial Discriminator Policy (raw)")
    display(res)

    _, res_ed = evaluate_majority_vote(
        model_dfs=d, answer_col = 'ED_consensus', label_col = 'answer_letter'
    )
    print(f"\n{label}: Consensus Game Policy (ED_consensus)")
    display(res_ed)

    print(f"\n{label}: Alignment Analysis (Init Discriminator)")
    display(evaluate_answer_alignment(
        d, answer_col = 'disc_init_refine_answer', label_col = 'answer_letter'
    ))

# Store for later use in statistical tests
combined_6_init_dis, results_6_init_dis = evaluate_majority_vote(
    model_dfs = discriminator_6dict, answer_col = 'disc_init_answer', label_col = 'answer_letter'
)
combined_6_ED, results_6_ED = evaluate_majority_vote(
    model_dfs = discriminator_6dict, answer_col = 'ED_consensus', label_col = 'answer_letter'
)

6-Model: Initial Discriminator Policy (raw)


,Metric,Count,Accuracy
0,Total Questions,10000,100%
1,Qwen25_7B Accuracy,4820,48.20%
2,Mistral_7B Accuracy,4772,47.72%
3,Gemma_7B Accuracy,5020,50.20%
4,Yi_9B Accuracy,5266,52.66%
5,Granite_8B Accuracy,5524,55.24%
6,Zephyr_7B Accuracy,5003,50.03%
7,Majority Vote (>3 models) Accuracy,4863,48.63%



6-Model: Consensus Game Policy (ED_consensus)


,Metric,Count,Accuracy
0,Total Questions,10000,100%
1,Qwen25_7B Accuracy,4820,48.20%
2,Mistral_7B Accuracy,4772,47.72%
3,Gemma_7B Accuracy,5020,50.20%
4,Yi_9B Accuracy,5266,52.66%
5,Granite_8B Accuracy,5524,55.24%
6,Zephyr_7B Accuracy,5003,50.03%
7,Majority Vote (>3 models) Accuracy,4863,48.63%



6-Model: Alignment Analysis (Init Discriminator)


...

,Metric,Count,Accuracy
0,Total Questions,10000,100%
1,Qwen25_7B Accuracy,4820,48.20%
2,Mistral_7B Accuracy,4772,47.72%
3,Gemma_7B Accuracy,5020,50.20%
4,Yi_9B Accuracy,5266,52.66%
5,Granite_8B Accuracy,5524,55.24%
6,Zephyr_7B Accuracy,5003,50.03%
7,DeepSeek_LLM Accuracy,4639,46.39%
8,DeepSeek_R1_Qwen Accuracy,4830,48.30%
9,Majority Vote (>4 models) Accuracy,4928,49.28%



8-Model: Consensus Game Policy (ED_consensus)


,Metric,Count,Accuracy
0,Total Questions,10000,100%
1,Qwen25_7B Accuracy,4820,48.20%
2,Mistral_7B Accuracy,4772,47.72%
3,Gemma_7B Accuracy,5020,50.20%
4,Yi_9B Accuracy,5266,52.66%
5,Granite_8B Accuracy,5524,55.24%
6,Zephyr_7B Accuracy,5003,50.03%
7,DeepSeek_LLM Accuracy,4639,46.39%
8,DeepSeek_R1_Qwen Accuracy,4830,48.30%
9,Majority Vote (>4 models) Accuracy,4928,49.28%



8-Model: Alignment Analysis (Init Discriminator)


,Metric,Count,Accuracy
0,Total Samples,10000,100%
1,Qwen25_7B Accuracy,4820,48.20%
2,Mistral_7B Accuracy,4772,47.72%
3,Gemma_7B Accuracy,5020,50.20%
4,Yi_9B Accuracy,5266,52.66%
5,Granite_8B Accuracy,5524,55.24%
6,Zephyr_7B Accuracy,5003,50.03%
7,DeepSeek_LLM Accuracy,4639,46.39%
8,DeepSeek_R1_Qwen Accuracy,4830,48.30%
9,Unanimous Agreement Rate,952,9.52%


## Single PEG Run (3 Models)

In [31]:
discriminator_3dict_update = batch_update_discriminators(
    discriminator_3dict,
    choice_df=df_qwen,
    batch_size=8,
    T_steps=11,
    learning_rate=0.1
)

combined_3_peg, results_3_peg = evaluate_majority_vote(
    discriminator_3dict_update,
    answer_col = 'updated_answer_letter',
    label_col = 'answer_letter'
)
print("=== 3-Model: PEG Updated Policy ===")
display(results_3_peg)

print("\n=== 3-Model: Alignment Analysis (After PEG) ===")
display(evaluate_answer_alignment(
    discriminator_3dict_update,
    answer_col = 'updated_answer_letter',
    label_col = 'answer_letter'
))

=== 3-Model: PEG Updated Policy ===


,Metric,Count,Accuracy
0,Total Questions,10000,100%
1,Qwen25_7B Accuracy,4794,47.94%
2,Mistral_7B Accuracy,4755,47.55%
3,Gemma_7B Accuracy,4967,49.67%
4,Majority Vote (>1 models) Accuracy,4905,49.05%



=== 3-Model: Alignment Analysis (After PEG) ===


,Metric,Count,Accuracy
0,Total Samples,10000,100%
1,Qwen25_7B Accuracy,4794,47.94%
2,Mistral_7B Accuracy,4755,47.55%
3,Gemma_7B Accuracy,4967,49.67%
4,Unanimous Agreement Rate,8265,82.65%
5,Unanimous & Correct Rate,3938,39.38%
6,Accuracy Given Unanimous Agreement,3938,47.65%


## Iteration Sweep (3 Models, T_steps 0 -> 49)

In [32]:
majority_vote_accuracies_3 = []

for t_step_val in range(50):
    disc_update = batch_update_discriminators(
        discriminator_3dict,
        choice_df = df_qwen,
        batch_size = 8,
        T_steps = t_step_val,
        learning_rate = 0.1
    )
    _, results = evaluate_majority_vote(
        disc_update,
        answer_col = 'updated_answer_letter',
        label_col = 'answer_letter'
    )
    majority_vote_accuracies_3.append(results.iloc[-1]['Accuracy'])

plot_df_3 = pd.DataFrame({
    'Iteration Number (T_steps)': list(range(50)),
    'Majority Vote Accuracy (3 Models)': majority_vote_accuracies_3
})
display(plot_df_3)

,Iteration Number (T_steps),Majority Vote Accuracy (3 Models)
0,0,49.23%
1,1,49.22%
2,2,49.17%
3,3,49.11%
4,4,49.09%
5,5,49.12%
6,6,49.09%
7,7,49.08%
8,8,49.10%
9,9,49.14%


## Single PEG Run (6/7/8 Models)

In [40]:
# ── 6-Model PEG ───────────────────────────────────────────────────────────────
discriminator_6dict_update = batch_update_discriminators(
    discriminator_6dict,
    choice_df = df_qwen,
    batch_size = 8,
    T_steps = 8,
    learning_rate = 0.1
)
combined_6_peg, results_6_peg = evaluate_majority_vote(
    discriminator_6dict_update,
    answer_col = 'updated_answer_letter',
    label_col = 'answer_letter'
)
print("6-Model: PEG Updated Policy")
display(results_6_peg)
print("\n6-Model: Alignment Analysis (After PEG)")
display(evaluate_answer_alignment(
    discriminator_6dict_update,
    answer_col = 'updated_answer_letter',
    label_col =' answer_letter'
))

# ── 7-Model PEG ───────────────────────────────────────────────────────────────
discriminator_7dict_update = batch_update_discriminators(
    discriminator_7dict,
    choice_df = df_qwen,
    batch_size = 8,
    T_steps = 8,
    learning_rate = 0.1
)
combined_7_peg, results_7_peg = evaluate_majority_vote(
    discriminator_7dict_update,
    answer_col = 'updated_answer_letter',
    label_col = 'answer_letter'
)
print("\n7-Model: PEG Updated Policy")
display(results_7_peg)
print("\n7-Model: Alignment Analysis (After PEG)")
display(evaluate_answer_alignment(
    discriminator_7dict_update,
    answer_col = 'updated_answer_letter',
    label_col = 'answer_letter'
))

# ── 8-Model PEG ───────────────────────────────────────────────────────────────
discriminator_8dict_update = batch_update_discriminators(
    discriminator_8dict,
    choice_df = df_qwen,
    batch_size = 8,
    T_steps = 8,
    learning_rate = 0.1
)
combined_8_peg, results_8_peg = evaluate_majority_vote(
    discriminator_8dict_update,
    answer_col = 'updated_answer_letter',
    label_col = 'answer_letter'
)
print("\n8-Model: PEG Updated Policy")
display(results_8_peg)
print("\n8-Model: Alignment Analysis (After PEG)")
display(evaluate_answer_alignment(
    discriminator_8dict_update,
    answer_col = 'updated_answer_letter',
    label_col = 'answer_letter'
))

6-Model: PEG Updated Policy


,Metric,Count,Accuracy
0,Total Questions,10000,100%
1,Qwen25_7B Accuracy,4853,48.53%
2,Mistral_7B Accuracy,4781,47.81%
3,Gemma_7B Accuracy,5033,50.33%
4,Yi_9B Accuracy,5224,52.24%
5,Granite_8B Accuracy,5517,55.17%
6,Zephyr_7B Accuracy,5011,50.11%
7,Majority Vote (>3 models) Accuracy,4743,47.43%



6-Model: Alignment Analysis (After PEG)


,Metric,Count,Accuracy
0,Total Samples,10000,100%
1,Qwen25_7B Accuracy,4853,48.53%
2,Mistral_7B Accuracy,4781,47.81%
3,Gemma_7B Accuracy,5033,50.33%
4,Yi_9B Accuracy,5224,52.24%
5,Granite_8B Accuracy,5517,55.17%
6,Zephyr_7B Accuracy,5011,50.11%
7,Unanimous Agreement Rate,1254,12.54%
8,Unanimous & Correct Rate,716,7.16%
9,Accuracy Given Unanimous Agreement,716,57.10%



7-Model: PEG Updated Policy


...

,Metric,Count,Accuracy
0,Total Samples,10000,100%
1,Qwen25_7B Accuracy,4814,48.14%
2,Mistral_7B Accuracy,4700,47.00%
3,Gemma_7B Accuracy,5029,50.29%
4,Yi_9B Accuracy,5219,52.19%
5,Granite_8B Accuracy,5510,55.10%
6,Zephyr_7B Accuracy,5003,50.03%
7,DeepSeek_LLM Accuracy,4617,46.17%
8,Unanimous Agreement Rate,1002,10.02%
9,Unanimous & Correct Rate,544,5.44%



8-Model: PEG Updated Policy


,Metric,Count,Accuracy
0,Total Questions,10000,100%
1,Qwen25_7B Accuracy,4690,46.90%
2,Mistral_7B Accuracy,4729,47.29%
3,Gemma_7B Accuracy,4902,49.02%
4,Yi_9B Accuracy,5144,51.44%
5,Granite_8B Accuracy,5569,55.69%
6,Zephyr_7B Accuracy,4883,48.83%
7,DeepSeek_LLM Accuracy,4640,46.40%
8,DeepSeek_R1_Qwen Accuracy,4696,46.96%
9,Majority Vote (>4 models) Accuracy,4627,46.27%



8-Model: Alignment Analysis (After PEG)


,Metric,Count,Accuracy
0,Total Samples,10000,100%
1,Qwen25_7B Accuracy,4690,46.90%
2,Mistral_7B Accuracy,4729,47.29%
3,Gemma_7B Accuracy,4902,49.02%
4,Yi_9B Accuracy,5144,51.44%
5,Granite_8B Accuracy,5569,55.69%
6,Zephyr_7B Accuracy,4883,48.83%
7,DeepSeek_LLM Accuracy,4640,46.40%
8,DeepSeek_R1_Qwen Accuracy,4696,46.96%
9,Unanimous Agreement Rate,829,8.29%


## Iteration Sweep (6 Models, T_steps 0 -> 49)

In [34]:
results_sweep = {3: [], 6: [], 7: [], 8: []}
configs_sweep = [
    (3, discriminator_3dict),
    (6, discriminator_6dict),
    (7, discriminator_7dict),
    (8, discriminator_8dict),
]

for n_models, disc_dict in configs_sweep:
    print(f"Running sweep for {n_models}-model configuration...")
    for t_step_val in range(50):
        disc_update = batch_update_discriminators(
            disc_dict,
            choice_df = df_qwen,
            batch_size = 8,
            T_steps = t_step_val,
            learning_rate = 0.1
        )
        _, res = evaluate_majority_vote(
            disc_update,
            answer_col = 'updated_answer_letter',
            label_col = 'answer_letter'
        )
        results_sweep[n_models].append(res.iloc[-1]['Accuracy'])

plot_df_all = pd.DataFrame({
    'Iteration Number (T_steps)': list(range(50)),
    'Majority Vote Accuracy (3 Models)': results_sweep[3],
    'Majority Vote Accuracy (6 Models)': results_sweep[6],
    'Majority Vote Accuracy (7 Models)': results_sweep[7],
    'Majority Vote Accuracy (8 Models)': results_sweep[8],
})
display(plot_df_all)

Running sweep for 3-model configuration...
Running sweep for 6-model configuration...
Running sweep for 7-model configuration...
Running sweep for 8-model configuration...


,Iteration Number (T_steps),Majority Vote Accuracy (3 Models),Majority Vote Accuracy (6 Models),Majority Vote Accuracy (7 Models),Majority Vote Accuracy (8 Models)
0,0,49.23%,48.63%,49.96%,49.28%
1,1,49.22%,48.84%,50.02%,49.28%
2,2,49.17%,48.81%,49.98%,49.13%
3,3,49.11%,48.71%,49.85%,49.09%
4,4,49.09%,48.52%,49.81%,48.77%
5,5,49.12%,48.22%,49.78%,48.36%
6,6,49.09%,47.86%,49.70%,48.02%
7,7,49.08%,45.85%,49.13%,47.08%
8,8,49.10%,47.43%,50.16%,46.27%
9,9,49.14%,46.96%,48.63%,46.50%


## Combined Comparison Table

In [41]:
def get_mv_accuracy(results_df):
    return results_df.iloc[-1]['Accuracy']

# ── Full comparison table ──────────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Configuration': [
        '3-Model: Init Discriminator (raw)',
        '3-Model: Init Discriminator (softmax-refined)',
        '3-Model: Consensus Game (ED_consensus)',
        '3-Model: PEG Updated (T=11)',
        '6-Model: Init Discriminator (raw)',
        '6-Model: Consensus Game (ED_consensus)',
        '6-Model: PEG Updated (T=8)',
        '7-Model: PEG Updated (T=8)',
        '8-Model: PEG Updated (T=8)',
    ],
    'Majority Vote Accuracy': [
        get_mv_accuracy(results_3_init_dis),
        get_mv_accuracy(results_3_init_refine),
        get_mv_accuracy(results_3_ED),
        get_mv_accuracy(results_3_peg),
        get_mv_accuracy(results_6_init_dis),
        get_mv_accuracy(results_6_ED),
        get_mv_accuracy(results_6_peg),
        get_mv_accuracy(results_7_peg),
        get_mv_accuracy(results_8_peg),
    ]
})
print("Full Configuration Comparison")
display(comparison)

# ── Scaling analysis: PEG accuracy vs. number of discriminators ───────────────
scaling = pd.DataFrame({
    'Number of Discriminators': [3, 6, 7, 8],
    'PEG Majority Vote Accuracy': [
        get_mv_accuracy(results_3_peg),
        get_mv_accuracy(results_6_peg),
        get_mv_accuracy(results_7_peg),
        get_mv_accuracy(results_8_peg),
    ]
})
print("\nScaling Analysis: PEG Accuracy vs. Number of Discriminators")
display(scaling)

Full Configuration Comparison


,Configuration,Majority Vote Accuracy
0,3-Model: Init Discriminator (raw),49.23%
1,3-Model: Init Discriminator (softmax-refined),49.23%
2,3-Model: Consensus Game (ED_consensus),49.23%
3,3-Model: PEG Updated (T=11),49.05%
4,6-Model: Init Discriminator (raw),48.63%
5,6-Model: Consensus Game (ED_consensus),48.63%
6,6-Model: PEG Updated (T=8),47.43%
7,7-Model: PEG Updated (T=8),50.16%
8,8-Model: PEG Updated (T=8),46.27%



Scaling Analysis: PEG Accuracy vs. Number of Discriminators


,Number of Discriminators,PEG Majority Vote Accuracy
0,3,49.05%
1,6,47.43%
2,7,50.16%
3,8,46.27%
